In [1]:
# !pip install httpx
# !pip install scrapy

In [2]:
import os
import sys

import arcpy

import json
import asyncio
import httpx

# set workspace folder
workspace = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks'
arcpy.env.workspace = workspace

# set sys path for custom folder module

# Append the directory to the Python path
sys.path.append(workspace)

from utils import check_files_exist

In [3]:
# folder to create and folder input path for BBOX - envelope
folder_name = 'input_shp'

# directly run from separate module
folder_path = check_files_exist.create_folder(workspace,folder_name)
print(folder_path)

Folder 'input_shp' already exists in workspace: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\input_shp


In [4]:
# shapefile name for input, to get boundary aoi
shp_name = 'rectangle_aoi.shp'
geometry_type = 'POLYGON'

In [5]:
file_path = os.path.join(folder_path,shp_name)

if not arcpy.Exists(file_path):
    # Create a new shapefile using CreateFeatureclass_management
    arcpy.management.CreateFeatureclass(
        out_path=folder_path,
        out_name=shp_name,
        geometry_type="POLYGON",
        template=None,
        has_m="DISABLED",
        has_z="DISABLED",
        spatial_reference='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 10000;-100000 10000;8.98315284119521E-09;0.001;0.001;IsHighPrecision',
        config_keyword="",
        spatial_grid_1=0,
        spatial_grid_2=0,
        spatial_grid_3=0,
        out_alias=""
    )
else:
    print('file rectangle, already there')

file rectangle, already there


In [6]:
# description, properties of layer
desc = arcpy.Describe(file_path)

In [7]:
# envelope bbox for arcgis rest api
envelope = desc.extent

In [8]:
# print(envelope)
xmin, ymin, xmax, ymax = envelope.XMin, envelope.YMin, envelope.XMax, envelope.YMax

In [9]:
check_files_exist.create_folder(workspace,'input_json')

Folder 'input_json' already exists in workspace: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks


'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\input_json'

In [10]:
# make a new name json, testing only
json_file_name = 'bbox.json'

# adding path and defined, use hardcoded instead
json_file_path = workspace + "\\" + 'input_json' + "\\" + json_file_name

In [52]:
data = {}
data['xmin'] = xmin
data['ymin'] = ymin
data['xmax'] = xmax
data['ymax'] = ymax


# dictionary to json file
with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file)

# '{xmin:107.593163470013,ymin:-7.13806689350134,xmax:107.685427020972,ymax:-7.06496818910933}' 

In [64]:
str(dict_data)

"{'xmin': 112.3266935978204, 'ymin': -1.4757927737340992, 'xmax': 115.48910668439328, 'ymax': 0.654674989852083}"

In [67]:
## AFTER EDITED in spider, to put in spatial envelope, let's run the command scrapy and check
# RUN THIS COMMAND IN ROOT, in the folder that has scrapy.cfg file
# scrapy crawl webgis_klhk_spider -O bbox_oid_list.json (or any other name .json)
# bbox_oid_list.json is  objectids that needed for request

In [11]:
# for testing only, this will be used in scrapy

# open the file of json and create dictionary object of it
with open(workspace + "\\" + 'bbox_oid_list.json', 'r' ) as oid_file:
    oid_dict = json.load(oid_file)

# oid_dict
print(len(oid_dict))

# flatten, to avoid nested for loop
flatten_dict_oid = {}
for i in oid_dict:
    for url, value in i.items():
        flatten_dict_oid[url] = value

# flatten_dict_oid
print(len(flatten_dict_oid))
        

for key, value in flatten_dict_oid.items():
    list_oids = flatten_dict_oid[key]['objectIds']
    if list_oids is not None:
        oid_name = flatten_dict_oid[key]['objectIdFieldName']
        chunk_size = 1000
        query_chunks = []
        for i in range(0, len(list_oids), chunk_size):
            chunk = list_oids[i:i + chunk_size]
            query = ' or '.join([f"{oid_name} = {str(oid)}" for oid in chunk])
            query_chunks.append(query)
        flatten_dict_oid[key]['query'] = query_chunks
        
fix_dict = {}
for key, value in flatten_dict_oid.items():
    if flatten_dict_oid[key].get('query') is not None:
#         a +=1
#         print(a)
#         print(key)
        fix_dict[key] = value
    
print(len(fix_dict))

35
35
33


In [13]:
# fix_dict['/server/rest/services/Time_Series/PL_2014/MapServer/0']['query']

In [15]:
# for i in fix_dict['/server/rest/services/Time_Series/PL_2014/MapServer/0']['query']:
#     print(i)
#     print('------- \n')

In [12]:
example_query_1000id = fix_dict['/server/rest/services/Time_Series/PL_2014/MapServer/0']['query'][0]

In [13]:
base_url = 'https://geoportal.menlhk.go.id/server/rest/services/Time_Series/PL_2014/MapServer/0/query'

In [15]:
params = {'where': 'objectid = 65605 or objectid = 83388 or objectid = 83516 or objectid = 90232 or objectid = 90553 or objectid = 91150 or objectid = 91355 or objectid = 91372 or objectid = 91373 or objectid = 91374 or objectid = 91380 or objectid = 91381 or objectid = 91384 or objectid = 91385 or objectid = 91386 or objectid = 91387 or objectid = 91388 or objectid = 91389 or objectid = 91390 or objectid = 91391 or objectid = 91392 or objectid = 91393 or objectid = 91394 or objectid = 91395 or objectid = 91396 or objectid = 91397 or objectid = 91398 or objectid = 91399 or objectid = 91401 or objectid = 91402 or objectid = 91403 or objectid = 91404 or objectid = 91405 or objectid = 91406 or objectid = 91407 or objectid = 91409 or objectid = 91410 or objectid = 91411 or objectid = 91412 or objectid = 91413 or objectid = 91414 or objectid = 91415 or objectid = 91416 or objectid = 91417 or objectid = 91418 or objectid = 91419 or objectid = 91420 or objectid = 91421 or objectid = 91422 or objectid = 91423 or objectid = 91424 or objectid = 91425 or objectid = 91426 or objectid = 91427 or objectid = 91428 or objectid = 91430 or objectid = 91431 or objectid = 91432 or objectid = 91433 or objectid = 91434 or objectid = 91435 or objectid = 91436 or objectid = 91437 or objectid = 91438 or objectid = 91439 or objectid = 91440 or objectid = 91441 or objectid = 91442 or objectid = 91443 or objectid = 91444 or objectid = 91445 or objectid = 91446 or objectid = 91447 or objectid = 91448 or objectid = 91449 or objectid = 91450 or objectid = 91451 or objectid = 91452 or objectid = 91453 or objectid = 91454 or objectid = 91455 or objectid = 91456 or objectid = 91457 or objectid = 91458 or objectid = 91459 or objectid = 91460 or objectid = 91461 or objectid = 91462 or objectid = 91463 or objectid = 91464 or objectid = 91465 or objectid = 91466 or objectid = 91467 or objectid = 91468 or objectid = 91469 or objectid = 91470 or objectid = 91471 or objectid = 91473 or objectid = 91474 or objectid = 91479 or objectid = 91480 or objectid = 91481 or objectid = 91482 or objectid = 91483 or objectid = 91484 or objectid = 91485 or objectid = 91486 or objectid = 91489 or objectid = 91490 or objectid = 91491 or objectid = 91492 or objectid = 91493 or objectid = 91494 or objectid = 91495 or objectid = 91496 or objectid = 91497 or objectid = 91498 or objectid = 91499 or objectid = 91500 or objectid = 91501 or objectid = 91509 or objectid = 91510 or objectid = 91511 or objectid = 91512 or objectid = 91513 or objectid = 91514 or objectid = 91515 or objectid = 91516 or objectid = 91517 or objectid = 91518 or objectid = 91519 or objectid = 91520 or objectid = 91521 or objectid = 91522 or objectid = 91523 or objectid = 91524 or objectid = 91525 or objectid = 91526 or objectid = 91527 or objectid = 91528 or objectid = 91529 or objectid = 91531 or objectid = 91532 or objectid = 91534 or objectid = 91535 or objectid = 91536 or objectid = 91538 or objectid = 91539 or objectid = 91540 or objectid = 91541 or objectid = 91542 or objectid = 91543 or objectid = 91546 or objectid = 91547 or objectid = 91548 or objectid = 91549 or objectid = 91550 or objectid = 91551 or objectid = 91553 or objectid = 91554 or objectid = 91555 or objectid = 91556 or objectid = 91557 or objectid = 91558 or objectid = 91559 or objectid = 91560 or objectid = 91561 or objectid = 91562 or objectid = 91563 or objectid = 91564 or objectid = 91565 or objectid = 91566 or objectid = 91567 or objectid = 91568 or objectid = 91569 or objectid = 91571 or objectid = 91572 or objectid = 91573 or objectid = 91574 or objectid = 91575 or objectid = 91576 or objectid = 91577 or objectid = 91578 or objectid = 91579 or objectid = 91580 or objectid = 91582 or objectid = 91583 or objectid = 91584 or objectid = 91585 or objectid = 91586 or objectid = 91587 or objectid = 91589 or objectid = 91597 or objectid = 91600 or objectid = 91601 or objectid = 91602 or objectid = 91603 or objectid = 91604 or objectid = 91605 or objectid = 91606 or objectid = 91607 or objectid = 91608 or objectid = 91609 or objectid = 91610 or objectid = 91611 or objectid = 91612 or objectid = 91613 or objectid = 91614 or objectid = 91615 or objectid = 91616 or objectid = 91617 or objectid = 91619 or objectid = 91620 or objectid = 91621 or objectid = 91622 or objectid = 91623 or objectid = 91624 or objectid = 91625 or objectid = 91626 or objectid = 91627 or objectid = 91628 or objectid = 91629 or objectid = 91631 or objectid = 91632 or objectid = 91633 or objectid = 91634 or objectid = 91635 or objectid = 91636 or objectid = 91637 or objectid = 91638 or objectid = 91639 or objectid = 91640 or objectid = 91642 or objectid = 91643 or objectid = 91644 or objectid = 91648 or objectid = 91650 or objectid = 91651 or objectid = 91652 or objectid = 91653 or objectid = 91654 or objectid = 91655 or objectid = 91656 or objectid = 91657 or objectid = 91658 or objectid = 91659 or objectid = 91660 or objectid = 91661 or objectid = 91666 or objectid = 91667 or objectid = 91668 or objectid = 91816 or objectid = 91941 or objectid = 91946 or objectid = 91947 or objectid = 91948 or objectid = 91950 or objectid = 91951 or objectid = 91952 or objectid = 91953 or objectid = 91954 or objectid = 91955 or objectid = 91956 or objectid = 91957 or objectid = 91958 or objectid = 91959 or objectid = 91960 or objectid = 91961 or objectid = 91962 or objectid = 91963 or objectid = 91965 or objectid = 91966 or objectid = 91967 or objectid = 91968 or objectid = 91969 or objectid = 91970 or objectid = 91971 or objectid = 91972 or objectid = 91973 or objectid = 91974 or objectid = 91975 or objectid = 91976 or objectid = 91977 or objectid = 91978 or objectid = 91979 or objectid = 91980 or objectid = 91981 or objectid = 91982 or objectid = 91983 or objectid = 91984 or objectid = 91985 or objectid = 91986 or objectid = 91988 or objectid = 91990 or objectid = 91991 or objectid = 91992 or objectid = 91993 or objectid = 91994 or objectid = 91995 or objectid = 91996 or objectid = 91997 or objectid = 91998 or objectid = 91999 or objectid = 92000 or objectid = 92001 or objectid = 92002 or objectid = 92003 or objectid = 92004 or objectid = 92006 or objectid = 92007 or objectid = 92008 or objectid = 92009 or objectid = 92010 or objectid = 92011 or objectid = 92012 or objectid = 92013 or objectid = 92014 or objectid = 92015 or objectid = 92016 or objectid = 92017 or objectid = 92018 or objectid = 92019 or objectid = 92021 or objectid = 92022 or objectid = 92023 or objectid = 92024 or objectid = 92026 or objectid = 92027 or objectid = 92028 or objectid = 92029 or objectid = 92030 or objectid = 92031 or objectid = 92032 or objectid = 92033 or objectid = 92034 or objectid = 92035 or objectid = 92036 or objectid = 92037 or objectid = 92038 or objectid = 92040 or objectid = 92041 or objectid = 92043 or objectid = 92044 or objectid = 92045 or objectid = 92046 or objectid = 92047 or objectid = 92050 or objectid = 92051 or objectid = 92052 or objectid = 92053 or objectid = 92054 or objectid = 92055 or objectid = 92056 or objectid = 92057 or objectid = 92058 or objectid = 92059 or objectid = 92060 or objectid = 92061 or objectid = 92062 or objectid = 92064 or objectid = 92065 or objectid = 92066 or objectid = 92067 or objectid = 92068 or objectid = 92069 or objectid = 92070 or objectid = 92071 or objectid = 92072 or objectid = 92073 or objectid = 92074 or objectid = 92075 or objectid = 92076 or objectid = 92077 or objectid = 92078 or objectid = 92079 or objectid = 92080 or objectid = 92081 or objectid = 92082 or objectid = 92083 or objectid = 92084 or objectid = 92085 or objectid = 92086 or objectid = 92087 or objectid = 92088 or objectid = 92090 or objectid = 92091 or objectid = 92093 or objectid = 92094 or objectid = 92095 or objectid = 92096 or objectid = 92097 or objectid = 92100 or objectid = 92101 or objectid = 92102 or objectid = 92104 or objectid = 92105 or objectid = 92106 or objectid = 92107 or objectid = 92108 or objectid = 92109 or objectid = 92110 or objectid = 92111 or objectid = 92112 or objectid = 92114 or objectid = 92115 or objectid = 92116 or objectid = 92117 or objectid = 92118 or objectid = 92119 or objectid = 92120 or objectid = 92121 or objectid = 92122 or objectid = 92123 or objectid = 92124 or objectid = 92125 or objectid = 92126 or objectid = 92127 or objectid = 92128 or objectid = 92129 or objectid = 92130 or objectid = 92131 or objectid = 92132 or objectid = 92133 or objectid = 92134 or objectid = 92135 or objectid = 92136 or objectid = 92137 or objectid = 92138 or objectid = 92139 or objectid = 92140 or objectid = 92141 or objectid = 92142 or objectid = 92143 or objectid = 92144 or objectid = 92145 or objectid = 92146 or objectid = 92147 or objectid = 92148 or objectid = 92149 or objectid = 92151 or objectid = 92152 or objectid = 92153 or objectid = 92154 or objectid = 92155 or objectid = 92156 or objectid = 92157 or objectid = 92158 or objectid = 92159 or objectid = 92160 or objectid = 92161 or objectid = 92162 or objectid = 92163 or objectid = 92164 or objectid = 92165 or objectid = 92166 or objectid = 92167 or objectid = 92168 or objectid = 92169 or objectid = 92170 or objectid = 92171 or objectid = 92172 or objectid = 92173 or objectid = 92174 or objectid = 92175 or objectid = 92176 or objectid = 92177 or objectid = 92178 or objectid = 92179 or objectid = 92180 or objectid = 92181 or objectid = 92182 or objectid = 92183 or objectid = 92184 or objectid = 92185 or objectid = 92186 or objectid = 92187 or objectid = 92188 or objectid = 92190 or objectid = 92191 or objectid = 92192 or objectid = 92193 or objectid = 92194 or objectid = 92195 or objectid = 92196 or objectid = 92197 or objectid = 92199 or objectid = 92200 or objectid = 92201 or objectid = 92203 or objectid = 92204 or objectid = 92205 or objectid = 92206 or objectid = 92207 or objectid = 92208 or objectid = 92209 or objectid = 92210 or objectid = 92211 or objectid = 92212 or objectid = 92213 or objectid = 92214 or objectid = 92215 or objectid = 92216 or objectid = 92217 or objectid = 92218 or objectid = 92220 or objectid = 92221 or objectid = 92222 or objectid = 92223 or objectid = 92224 or objectid = 92226 or objectid = 92227 or objectid = 92228 or objectid = 92229 or objectid = 92230 or objectid = 92231 or objectid = 92232 or objectid = 92233 or objectid = 92234 or objectid = 92235 or objectid = 92236 or objectid = 92237 or objectid = 92238 or objectid = 92239 or objectid = 92240 or objectid = 92241 or objectid = 92242 or objectid = 92243 or objectid = 92244 or objectid = 92246 or objectid = 92247 or objectid = 92248 or objectid = 92249 or objectid = 92250 or objectid = 92251 or objectid = 92252 or objectid = 92253 or objectid = 92255 or objectid = 92257 or objectid = 92258 or objectid = 92259 or objectid = 92260 or objectid = 92261 or objectid = 92262 or objectid = 92263 or objectid = 92264 or objectid = 92265 or objectid = 92266 or objectid = 92267 or objectid = 92268 or objectid = 92269 or objectid = 92270 or objectid = 92271 or objectid = 92272 or objectid = 92273 or objectid = 92274 or objectid = 92275 or objectid = 92276 or objectid = 92277 or objectid = 92278 or objectid = 92279 or objectid = 92280 or objectid = 92281 or objectid = 92282 or objectid = 92283 or objectid = 92285 or objectid = 92286 or objectid = 92287 or objectid = 92288 or objectid = 92289 or objectid = 92290 or objectid = 92291 or objectid = 92292 or objectid = 92293 or objectid = 92294 or objectid = 92295 or objectid = 92296 or objectid = 92297 or objectid = 92298 or objectid = 92299 or objectid = 92300 or objectid = 92302 or objectid = 92303 or objectid = 92304 or objectid = 92305 or objectid = 92306 or objectid = 92307 or objectid = 92308 or objectid = 92309 or objectid = 92310 or objectid = 92311 or objectid = 92312 or objectid = 92313 or objectid = 92315 or objectid = 92316 or objectid = 92317 or objectid = 92318 or objectid = 92319 or objectid = 92320 or objectid = 92321 or objectid = 92322 or objectid = 92323 or objectid = 92325 or objectid = 92326 or objectid = 92328 or objectid = 92329 or objectid = 92330 or objectid = 92331 or objectid = 92332 or objectid = 92333 or objectid = 92334 or objectid = 92335 or objectid = 92336 or objectid = 92337 or objectid = 92338 or objectid = 92339 or objectid = 92340 or objectid = 92341 or objectid = 92342 or objectid = 92343 or objectid = 92344 or objectid = 92345 or objectid = 92346 or objectid = 92347 or objectid = 92348 or objectid = 92349 or objectid = 92350 or objectid = 92351 or objectid = 92352 or objectid = 92353 or objectid = 92355 or objectid = 92357 or objectid = 92358 or objectid = 92359 or objectid = 92360 or objectid = 92361 or objectid = 92362 or objectid = 92363 or objectid = 92364 or objectid = 92365 or objectid = 92366 or objectid = 92367 or objectid = 92368 or objectid = 92369 or objectid = 92370 or objectid = 92371 or objectid = 92372 or objectid = 92373 or objectid = 92374 or objectid = 92376 or objectid = 92377 or objectid = 92378 or objectid = 92379 or objectid = 92380 or objectid = 92381 or objectid = 92382 or objectid = 92383 or objectid = 92384 or objectid = 92385 or objectid = 92386 or objectid = 92387 or objectid = 92388 or objectid = 92389 or objectid = 92390 or objectid = 92391 or objectid = 92392 or objectid = 92393 or objectid = 92394 or objectid = 92395 or objectid = 92396 or objectid = 92397 or objectid = 92398 or objectid = 92399 or objectid = 92400 or objectid = 92401 or objectid = 92402 or objectid = 92403 or objectid = 92404 or objectid = 92405 or objectid = 92406 or objectid = 92407 or objectid = 92408 or objectid = 92409 or objectid = 92410 or objectid = 92411 or objectid = 92412 or objectid = 92413 or objectid = 92414 or objectid = 92415 or objectid = 92416 or objectid = 92417 or objectid = 92419 or objectid = 92420 or objectid = 92421 or objectid = 92422 or objectid = 92423 or objectid = 92424 or objectid = 92425 or objectid = 92426 or objectid = 92427 or objectid = 92428 or objectid = 92429 or objectid = 92430 or objectid = 92431 or objectid = 92432 or objectid = 92433 or objectid = 92434 or objectid = 92435 or objectid = 92436 or objectid = 92437 or objectid = 92438 or objectid = 92439 or objectid = 92440 or objectid = 92441 or objectid = 92442 or objectid = 92445 or objectid = 92448 or objectid = 92449 or objectid = 92451 or objectid = 92452 or objectid = 92454 or objectid = 92455 or objectid = 92456 or objectid = 92457 or objectid = 92458 or objectid = 92459 or objectid = 92460 or objectid = 92461 or objectid = 92462 or objectid = 92463 or objectid = 92464 or objectid = 92465 or objectid = 92466 or objectid = 92467 or objectid = 92468 or objectid = 92469 or objectid = 92470 or objectid = 92471 or objectid = 92472 or objectid = 92473 or objectid = 92474 or objectid = 92475 or objectid = 92476 or objectid = 92477 or objectid = 92478 or objectid = 92479 or objectid = 92480 or objectid = 92481 or objectid = 92482 or objectid = 92483 or objectid = 92484 or objectid = 92485 or objectid = 92486 or objectid = 92487 or objectid = 92488 or objectid = 92489 or objectid = 92490 or objectid = 92491 or objectid = 92492 or objectid = 92493 or objectid = 92494 or objectid = 92495 or objectid = 92496 or objectid = 92498 or objectid = 92499 or objectid = 92500 or objectid = 92501 or objectid = 92502 or objectid = 92503 or objectid = 92504 or objectid = 92505 or objectid = 92506 or objectid = 92507 or objectid = 92508 or objectid = 92509 or objectid = 92510 or objectid = 92511 or objectid = 92512 or objectid = 92513 or objectid = 92514 or objectid = 92515 or objectid = 92516 or objectid = 92517 or objectid = 92518 or objectid = 92519 or objectid = 92520 or objectid = 92522 or objectid = 92523 or objectid = 92524 or objectid = 92525 or objectid = 92526 or objectid = 92527 or objectid = 92528 or objectid = 92529 or objectid = 92530 or objectid = 92531 or objectid = 92532 or objectid = 92533 or objectid = 92534 or objectid = 92535 or objectid = 92536 or objectid = 92537 or objectid = 92538 or objectid = 92539 or objectid = 92540 or objectid = 92541 or objectid = 92542 or objectid = 92543 or objectid = 92544 or objectid = 92545 or objectid = 92546 or objectid = 92547 or objectid = 92548 or objectid = 92549 or objectid = 92550 or objectid = 92551 or objectid = 92552 or objectid = 92565 or objectid = 92566 or objectid = 92567 or objectid = 92598 or objectid = 92599 or objectid = 92601 or objectid = 92602 or objectid = 92603 or objectid = 92604 or objectid = 92605 or objectid = 92606 or objectid = 92607 or objectid = 92608 or objectid = 92609 or objectid = 92610 or objectid = 92611 or objectid = 92612 or objectid = 92613 or objectid = 92614 or objectid = 92615 or objectid = 92616 or objectid = 92617 or objectid = 92618 or objectid = 92619 or objectid = 92620 or objectid = 92621 or objectid = 92622 or objectid = 92624 or objectid = 92625 or objectid = 92626 or objectid = 92627 or objectid = 92628 or objectid = 92629 or objectid = 92630 or objectid = 92631 or objectid = 92632 or objectid = 92633 or objectid = 92634 or objectid = 92635 or objectid = 92636 or objectid = 92637 or objectid = 92638 or objectid = 92639 or objectid = 92640 or objectid = 92641 or objectid = 92642 or objectid = 92643 or objectid = 92644 or objectid = 92645 or objectid = 92646 or objectid = 92647 or objectid = 92648 or objectid = 92649 or objectid = 92651 or objectid = 92652 or objectid = 92653 or objectid = 92654 or objectid = 92655 or objectid = 92656 or objectid = 92657 or objectid = 92658 or objectid = 92659 or objectid = 92660 or objectid = 92661 or objectid = 92662 or objectid = 92663 or objectid = 92664 or objectid = 92665 or objectid = 92666 or objectid = 92667 or objectid = 92668 or objectid = 92669 or objectid = 92670 or objectid = 92671 or objectid = 92672 or objectid = 92673 or objectid = 92674 or objectid = 92675 or objectid = 92676 or objectid = 92677 or objectid = 92678 or objectid = 92679 or objectid = 92680 or objectid = 92681 or objectid = 92682 or objectid = 92683 or objectid = 92684 or objectid = 92685 or objectid = 92686 or objectid = 92687 or objectid = 92688 or objectid = 92689 or objectid = 92690 or objectid = 92691 or objectid = 92692 or objectid = 92693 or objectid = 92694 or objectid = 92695 or objectid = 92696 or objectid = 92697 or objectid = 92698 or objectid = 92699 or objectid = 92700 or objectid = 92701 or objectid = 92702 or objectid = 92703 or objectid = 92704 or objectid = 92705 or objectid = 92706 or objectid = 92707 or objectid = 92708 or objectid = 92709 or objectid = 92710 or objectid = 92711 or objectid = 92712 or objectid = 92713 or objectid = 92714 or objectid = 92715 or objectid = 92716 or objectid = 92717 or objectid = 92718 or objectid = 92719 or objectid = 92720 or objectid = 92721 or objectid = 92722 or objectid = 92723 or objectid = 92724 or objectid = 92725 or objectid = 92726 or objectid = 92727 or objectid = 92728 or objectid = 92729 or objectid = 92730 or objectid = 92731 or objectid = 92732 or objectid = 92733 or objectid = 92734 or objectid = 92735 or objectid = 92736 or objectid = 92737 or objectid = 92738 or objectid = 92739 or objectid = 92740 or objectid = 92741 or objectid = 92742 or objectid = 92743 or objectid = 92744 or objectid = 92745 or objectid = 92746 or objectid = 92747 or objectid = 92748 or objectid = 92749 or objectid = 92750 or objectid = 92751 or objectid = 92752 or objectid = 92753 or objectid = 92754 or objectid = 92755 or objectid = 92756 or objectid = 92757 or objectid = 92758 or objectid = 92759 or objectid = 92760 or objectid = 92761 or objectid = 92762 or objectid = 92771 or objectid = 92772 or objectid = 92773 or objectid = 92774 or objectid = 92775 or objectid = 92776 or objectid = 92852 or objectid = 92895 or objectid = 92896 or objectid = 92897 or objectid = 92898 or objectid = 92899 or objectid = 92900 or objectid = 92901 or objectid = 92902 or objectid = 92903 or objectid = 92904 or objectid = 92905 or objectid = 92906',
'text': '',
'objectIds': '',
'time': '',
'timeRelation': 'esriTimeRelationOverlaps',
'geometry': '',
'geometryType': 'esriGeometryEnvelope',
'inSR': '',
'spatialRel': 'esriSpatialRelIntersects',
'distance': '',
'units': 'esriSRUnit_Foot',
'relationParam': '',
'outFields': '*',
'returnGeometry': 'true',
'returnTrueCurves': 'false',
'maxAllowableOffset': '',
'geometryPrecision': '',
'outSR': '',
'havingClause': '',
'returnIdsOnly': 'false',
'returnCountOnly': 'false',
'orderByFields': '',
'groupByFieldsForStatistics':  '',
'outStatistics': '',
'returnZ': 'false',
'returnM': 'false',
'gdbVersion': '',
'historicMoment': '',
'returnDistinctValues': 'false',
'resultOffset': '',
'resultRecordCount': '',
'returnExtentOnly': 'false',
'sqlFormat': 'none',
'datumTransformation': '',
'parameterValues': '',
'rangeValues': '',
'quantizationParameters': '',
'featureEncoding': 'esriDefault',
'f': 'geojson',
    }

In [163]:
### ARCPY SPATIAL ANALYSIS after DOWNLOADING FINISH and GET THE DATA NEEDED ####
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.listMaps()[0]  # assumes data to be added to first map listed

workspace_scratch = r'Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb'
arcpy.env.workspace = workspace_scratch

gdb_location_lc = r'Z:\gdrive_treeo\GIS_data\LC MoEF 1990-2020\Landcover 1990-2020.gdb'


In [165]:
feature_input = "LC_MoEF_1990_2020_Indonesia_fix"

# map.addLayer(feature_input)

arcpy.management.MakeFeatureLayer(gdb_location_lc+"\\"+feature_input, feature_input) 


<Result 'LC_MoEF_1990_2020_Indonesia_fix'>

In [168]:
clip_shp_path = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\input_shp\rectangle_aoi.shp'
clip_shp_name = 'aoi_bbox_envelope'

arcpy.management.MakeFeatureLayer(clip_shp_path,
                                 clip_shp_name)

<Result 'aoi_bbox_envelope'>

In [169]:
arcpy.analysis.Clip(
    in_features=feature_input,
    clip_features=clip_shp_name,
    out_feature_class=workspace_scratch+"\\"+'AOI_LC_1990_2020_MoEF',
    cluster_tolerance=None
)

<Result 'Z:\\GIS_ArcGISPro\\yt_DEMO\\MyProject\\MyProject.gdb\\AOI_LC_1990_2020_MoEF'>

In [116]:
# !pip install geopandas

In [152]:
# import geopandas as gpd

# gdf = gpd.read_file(workspace+"\\"+'example_queries.json')
# gdf.to_file(workspace+"\\"+'example_queries.shp')

In [153]:
# aprx = arcpy.mp.ArcGISProject("CURRENT")
# map = aprx.listMaps()[0]  # assumes data to be added to first map listed

# location_shp = os.path.join(workspace,'example_queries.shp')

# map.addDataFromPath(location_shp)